In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/001/sensor_data/2014_10_01-10_09_39/2014_10_01-10_09_39_ECG.csv
/kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/001/sensor_data/2014_10_02-10_56_44/2014_10_02-10_56_44_ECG.csv
/kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/001/sensor_data/2014_10_03-06_36_24/2014_10_03-06_36_24_ECG.csv
/kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/001/sensor_data/2014_10_04-06_34_57/2014_10_04-06_34_57_ECG.csv
/kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_01-12_35_54/2014_10_01-12_35_54_ECG.csv
/kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_01-20_29_57/20

In [2]:
!pip install -q kaggle

import os, warnings, random, math, glob, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression, HuberRegressor
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
SEARCH_ROOTS = ["/kaggle/input", "/content", "/content/drive", "/mnt", "."]
_candidates = []
for root in SEARCH_ROOTS:
    if os.path.isdir(root):
        _candidates += glob.glob(f"{root}/**/diabetes_subset_sensor_data", recursive=True)
assert _candidates, "Dataset not found — check it's attached."
DATA_DIR = os.path.dirname(sorted(_candidates, key=len)[0])
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR)

DATA_DIR: /kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data


In [4]:
WINDOW_SIZE   = 48
HORIZONS      = [3, 6, 12]   # t+15, t+30, t+60 (5-min sampling)
SENSOR_COLS   = ["HR", "Activity", "PeakAccel",
                 "recent_carbs", "recent_units", "cgm_roc", "cgm_accel"]  # SkinTemp excluded — dead sensor
SKIN_TEMP_MIN, SKIN_TEMP_MAX = -40.0, 60.0
TOLERANCE_SEC = 90
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10

def resolve(*parts):
    p = os.path.join(DATA_DIR, *parts)
    inner = os.path.join(p, parts[0])
    return inner if os.path.isdir(inner) else p

subsets = {"diabetes": {"sensor": resolve("diabetes_subset_sensor_data"),
                        "glucose": resolve("diabetes_subset_pictures-glucose-food-insulin")}}

all_subjects = []
for group, paths in subsets.items():
    for subj_id in sorted(os.listdir(paths["sensor"])):
        sensor_path  = os.path.join(paths["sensor"], subj_id)
        glucose_path = os.path.join(paths["glucose"], subj_id, "glucose.csv")
        if os.path.isdir(sensor_path) and os.path.exists(glucose_path):
            all_subjects.append({"id": f"{group}_{subj_id}", "group": group,
                                  "sensor_path": sensor_path, "glucose_csv": glucose_path})
print(f"Subjects found: {len(all_subjects)}")

Subjects found: 9


In [5]:
def load_glucose(glucose_csv):
    df = pd.read_csv(glucose_csv)
    df = df[df["type"] == "cgm"].copy()
    df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])
    return df.sort_values("datetime").reset_index(drop=True)[["datetime", "glucose"]]

def load_sensor(sensor_path):
    sensor_dir = os.path.join(sensor_path, "sensor_data")
    if not os.path.isdir(sensor_dir):
        return None
    all_dfs = []
    for session in sorted(os.listdir(sensor_dir)):
        fpath = os.path.join(sensor_dir, session, f"{session}_Summary.csv")
        if os.path.exists(fpath):
            all_dfs.append(pd.read_csv(fpath))
    if not all_dfs:
        return None
    combined = pd.concat(all_dfs, ignore_index=True)
    combined["datetime"] = pd.to_datetime(combined["Time"], format="%d/%m/%Y %H:%M:%S.%f", errors="coerce")
    combined = combined.dropna(subset=["datetime"]).sort_values("datetime")
    base_cols = ["HR", "Activity", "PeakAccel"]
    cols = ["datetime"] + [c for c in base_cols if c in combined.columns]
    return combined[cols].reset_index(drop=True)

def _rolling_2h_sum(event_times, event_vals, query_times):
    if len(event_times) == 0:
        return np.zeros(len(query_times), dtype=np.float64)
    ev_t = event_times.values.astype("datetime64[ns]")
    ev_v = np.asarray(event_vals, dtype=np.float64)
    csum = np.concatenate([[0.0], np.cumsum(ev_v)])
    q = query_times.values.astype("datetime64[ns]")
    lo = q - np.timedelta64(2, "h")
    l_idx = np.searchsorted(ev_t, lo, side="left")
    r_idx = np.searchsorted(ev_t, q, side="right")
    return csum[r_idx] - csum[l_idx]

def add_meal_insulin_features(merged_df, subject_dir):
    for fname, col, out in [("food.csv", "carbs", "recent_carbs"),
                            ("insulin.csv", "units", "recent_units")]:
        fpath = os.path.join(subject_dir, fname)
        merged_df[out] = 0.0
        if not os.path.exists(fpath):
            continue
        df = pd.read_csv(fpath)
        if not {"date", "time", col}.issubset(df.columns):
            continue
        df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
        df = df.dropna(subset=["datetime"]).sort_values("datetime")
        merged_df[out] = _rolling_2h_sum(df["datetime"], df[col], merged_df["datetime"])
    return merged_df

def add_cgm_derivative_features(merged_df):
    g = merged_df["glucose"]
    merged_df["cgm_roc"]   = g.diff().fillna(0.0)
    merged_df["cgm_accel"] = merged_df["cgm_roc"].diff().fillna(0.0)
    return merged_df

def align_glucose_sensor(glucose_df, sensor_df):
    merged = pd.merge_asof(
        glucose_df.sort_values("datetime"), sensor_df.sort_values("datetime"),
        on="datetime", tolerance=pd.Timedelta(seconds=TOLERANCE_SEC), direction="nearest")
    return merged.dropna(subset=["glucose"]).reset_index(drop=True)

In [6]:
subject_merged = {}
for s in all_subjects:
    gdf = load_glucose(s["glucose_csv"]); sdf = load_sensor(s["sensor_path"])
    if sdf is None or len(gdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    merged = align_glucose_sensor(gdf, sdf)
    merged = add_meal_insulin_features(merged, os.path.dirname(s["glucose_csv"]))
    merged = add_cgm_derivative_features(merged)
    subject_merged[s["id"]] = merged

subj_ids = list(subject_merged.keys())
print("Subjects with usable data:", len(subj_ids))

Subjects with usable data: 9


In [7]:
D_MODEL = 64

class S4DKernel(nn.Module):
    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1):
        super().__init__()
        log_dt = torch.rand(d_model) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.log_dt = nn.Parameter(log_dt)
        C = torch.randn(d_model, d_state, 2) * 0.5**0.5
        self.C = nn.Parameter(C)
        log_A_real = torch.log(0.5 * torch.ones(d_model, d_state))
        A_imag = math.pi * torch.arange(d_state).repeat(d_model, 1)
        self.log_A_real = nn.Parameter(log_A_real)
        self.A_imag = nn.Parameter(A_imag)

    def forward(self, L):
        dt = torch.exp(self.log_dt)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag
        C = torch.view_as_complex(self.C)
        dtA = A * dt.unsqueeze(-1)
        K = dtA.unsqueeze(-1) * torch.arange(L, device=A.device)
        C_scaled = C * (torch.exp(dtA) - 1) / A
        K = 2 * torch.einsum('hn,hnl->hl', C_scaled, torch.exp(K)).real
        return K

class S4Block(nn.Module):
    def __init__(self, d_model, d_state=64, dropout=0.2, bidirectional=True):
        super().__init__()
        self.kernel = S4DKernel(d_model, d_state)
        self.D = nn.Parameter(torch.randn(d_model))
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, H = x.shape
        residual = x
        x = self.norm(x)
        u = x.transpose(1, 2)
        k = self.kernel(L)
        k_f = torch.fft.rfft(k, n=2*L)
        u_f = torch.fft.rfft(u, n=2*L)
        y = torch.fft.irfft(u_f * k_f, n=2*L)[..., :L]
        y = y + u * self.D.unsqueeze(-1)
        y = y.transpose(1, 2)
        y = self.dropout(F.gelu(y))
        return residual + self.out_proj(y)

class FFNBlock(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return x + self.net(self.norm(x))

print("S4 blocks ready.")

S4 blocks ready.


In [8]:
class D1NAMODataset(Dataset):
    def __init__(self, records, augment=False, use_physics=False):
        self.records = records
        self.augment = augment
        self.use_physics = use_physics

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        cgm, sensor = r["cgm_window"].copy(), r["sensor_window"].copy()
        if self.augment:
            if np.random.rand() < 0.5:
                cgm = cgm + np.random.normal(0, 0.05, size=cgm.shape).astype(np.float32)
            if np.random.rand() < 0.3 and sensor.shape[1] > 0:
                col = np.random.randint(0, sensor.shape[1])
                sensor[:, col] = 0.0
        out = [torch.tensor(cgm, dtype=torch.float32),
              torch.tensor(sensor, dtype=torch.float32),
              torch.tensor(r["targets"], dtype=torch.float32)]
        if self.use_physics:
            out.append(torch.tensor(r["physics_pred"], dtype=torch.float32))
        return tuple(out)

print("D1NAMODataset ready.")

D1NAMODataset ready.


In [9]:
def fit_empirical_formula_v3(all_merged_train, h):
    X_list, y_list = [], []
    for merged in all_merged_train:
        g = merged["glucose"].values.astype(np.float64)
        n = len(g)
        for i in range(12, n - h):
            g_now   = g[i]
            d_3     = g[i] - g[i-3]
            d_6     = g[i] - g[i-6]
            d_12    = g[i] - g[i-12]
            roc_now = g[i] - g[i-1]
            X_list.append([g_now, roc_now, d_3, d_6, d_12])
            y_list.append(g[i + h])

    X = np.array(X_list); y = np.array(y_list)
    reg = HuberRegressor(epsilon=1.35, max_iter=500).fit(X, y)
    names = ["a_g", "a_roc", "a_d3", "a_d6", "a_d12"]
    coefs = dict(zip(names, reg.coef_))
    coefs["b"] = reg.intercept_
    return coefs

def empirical_predict_v3(g_hist, h, coefs):
    c = coefs
    g_now = g_hist[-1]
    roc_now = g_hist[-1] - g_hist[-2]
    d_3  = g_hist[-1] - g_hist[-4]
    d_6  = g_hist[-1] - g_hist[-7]
    d_12 = g_hist[-1] - g_hist[-13]
    return c["a_g"]*g_now + c["a_roc"]*roc_now + c["a_d3"]*d_3 + c["a_d6"]*d_6 + c["a_d12"]*d_12 + c["b"]

print("Empirical v3 formula ready.")

Empirical v3 formula ready.


In [10]:
def build_windows_hybrid_v3(merged_df, v3_coefs):
    sensor_cols = [c for c in SENSOR_COLS if c in merged_df.columns]
    gluc = merged_df["glucose"].values.astype(np.float32)
    max_h = max(HORIZONS)
    if len(gluc) < WINDOW_SIZE + max_h + 12:
        return []

    g_mean, g_std = float(gluc.mean()), float(gluc.std()) + 1e-8
    g_norm = (gluc - g_mean) / g_std

    sarr = merged_df[sensor_cols].copy()
    for c in sensor_cols:
        lo, hi = sarr[c].min(), sarr[c].max()
        sarr[c] = (sarr[c] - lo) / (hi - lo + 1e-8)
    sarr = sarr.fillna(0).values.astype(np.float32)

    recs = []
    for i in range(WINDOW_SIZE, len(gluc) - max_h):
        cw, sw = g_norm[i-WINDOW_SIZE:i], sarr[i-WINDOW_SIZE:i]
        tg = np.array([(gluc[i+h] - g_mean) / g_std for h in HORIZONS], dtype=np.float32)
        if np.isnan(cw).any() or np.isnan(tg).any():
            continue

        g_hist = gluc[i-12:i+1].astype(np.float64)
        phys_preds = []
        for h in HORIZONS:
            p = empirical_predict_v3(g_hist, h, v3_coefs[h])
            phys_preds.append((p - g_mean) / g_std)

        recs.append({"cgm_window": cw, "sensor_window": sw, "targets": tg,
                     "glucose_mean": g_mean, "glucose_std": g_std,
                     "physics_pred": np.array(phys_preds, dtype=np.float32)})
    return recs

print("build_windows_hybrid_v3 ready.")

build_windows_hybrid_v3 ready.


In [11]:
class D1NAMOPhysicsInformed_TunedConstrained(nn.Module):
    def __init__(self, d_model=D_MODEL, d_state=64, num_layers=4, dropout=0.2,
                 window_size=WINDOW_SIZE, n_sensors=None, n_targets=3, bidirectional=True,
                 correction_scales=(0.20, 0.15, 0.05)):
        super().__init__()
        n_ch = 1 + n_sensors
        self.register_buffer("correction_scales", torch.tensor(correction_scales, dtype=torch.float32))
        self.in_proj = nn.Sequential(nn.Linear(n_ch, d_model), nn.GELU())
        self.pos = nn.Parameter(torch.randn(1, window_size, d_model) * 0.02)
        self.layers = nn.ModuleList([
            nn.ModuleDict({"s4": S4Block(d_model, d_state, dropout, bidirectional),
                          "ffn": FFNBlock(d_model, d_model * 4, dropout)})
            for _ in range(num_layers)])
        self.norm_out = nn.LayerNorm(d_model)
        self.physics_proj = nn.Sequential(nn.Linear(n_targets, d_model), nn.GELU())
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model * 2 + d_model, d_model), nn.GELU(),
                          nn.Dropout(dropout), nn.Linear(d_model, 1))
            for _ in range(n_targets)])
        for h in self.heads:
            nn.init.zeros_(h[-1].weight); nn.init.zeros_(h[-1].bias)

    def forward(self, cgm, sensor, physics_pred):
        x = torch.cat([cgm.unsqueeze(-1), sensor], dim=-1)
        x = self.in_proj(x) + self.pos
        for layer in self.layers:
            x = layer["s4"](x); x = layer["ffn"](x)
        x = self.norm_out(x)
        pooled = torch.cat([x.mean(dim=1), x[:, -1, :]], dim=-1)
        phys_feat = self.physics_proj(physics_pred)
        combined = torch.cat([pooled, phys_feat], dim=-1)
        correction = torch.cat([h(combined) for h in self.heads], dim=-1)
        return physics_pred + self.correction_scales * torch.tanh(correction)


def weighted_horizon_loss(pred, target, weights=(1.0, 1.0, 2.0)):
    w = torch.tensor(weights, device=pred.device, dtype=pred.dtype)
    per_horizon = F.smooth_l1_loss(pred, target, beta=0.5, reduction='none')
    return (per_horizon * w).mean()


def run_epoch_weighted(m, loader, opt=None):
    m.train() if opt else m.eval()
    total = 0.0
    ctx = torch.enable_grad() if opt else torch.no_grad()
    with ctx:
        for cgm, sensor, y, phys in loader:
            cgm, sensor, y, phys = cgm.to(device), sensor.to(device), y.to(device), phys.to(device)
            if opt:
                opt.zero_grad()
                pred = m(cgm, sensor, phys)
                loss = weighted_horizon_loss(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
            else:
                pred = m(cgm, sensor, phys)
                loss = weighted_horizon_loss(pred, y)
            total += loss.item() * cgm.size(0)
    return total / len(loader.dataset)


def mard(y_true, y_pred):
    return float(np.mean(np.abs(y_pred - y_true) / y_true) * 100)

def clarke_zone(ref, pred):
    ref, pred = ref * 18.0182, pred * 18.0182
    n = len(ref)
    zones = np.full(n, "E", dtype="<U1")
    zone_a = (np.abs(pred - ref) <= 0.2 * ref) | ((ref < 70) & (pred < 70))
    zone_e = ((ref >= 180) & (pred <= 70)) | ((ref <= 70) & (pred >= 180))
    zone_c = ((ref >= 70) & (ref <= 290) & (pred >= ref + 110)) | \
             ((ref >= 130) & (ref <= 180) & (pred <= (7/5) * ref - 182))
    zone_d = ((ref > 240) & (pred >= 70) & (pred <= 180)) | \
             ((ref < 70) & (pred >= 70) & (pred <= 180))
    zone_b = ~zone_a & ~zone_e & ~zone_c & ~zone_d
    zones[zone_e] = "E"; zones[zone_c] = "C"; zones[zone_d] = "D"; zones[zone_b] = "B"; zones[zone_a] = "A"
    return zones

print("Tuned model + helpers ready.")

Tuned model + helpers ready.


In [13]:
def run_losocv_tuned(n_epochs=60, patience=12, ft_epochs=4, ft_lr_scale=0.03, ft_frac=0.3):
    all_preds, all_trues = [], []
    for held_out in subj_ids:
        train_subjs = [s for s in subj_ids if s != held_out]
        v3_coefs = {h: fit_empirical_formula_v3([subject_merged[s] for s in train_subjs], h) for h in HORIZONS}

        train_recs, val_recs = [], []
        for s in train_subjs:
            recs = build_windows_hybrid_v3(subject_merged[s], v3_coefs)
            split = int(len(recs) * 0.9)
            train_recs.extend(recs[:split]); val_recs.extend(recs[split:])
        test_recs_all = build_windows_hybrid_v3(subject_merged[held_out], v3_coefs)
        ft_split = max(WINDOW_SIZE, int(len(test_recs_all) * ft_frac))
        ft_recs, eval_recs = test_recs_all[:ft_split], test_recs_all[ft_split:]
        if len(eval_recs) < 5:
            continue

        n_sensors = len(train_recs[0]["sensor_window"][0])
        tr_loader = DataLoader(D1NAMODataset(train_recs, augment=True, use_physics=True),
                               batch_size=32, shuffle=True, drop_last=True)
        va_loader = DataLoader(D1NAMODataset(val_recs, augment=False, use_physics=True),
                               batch_size=32, shuffle=False)
        ft_loader = DataLoader(D1NAMODataset(ft_recs, augment=False, use_physics=True),
                               batch_size=16, shuffle=True)
        ev_loader = DataLoader(D1NAMODataset(eval_recs, augment=False, use_physics=True),
                               batch_size=32, shuffle=False)

        torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
        m = D1NAMOPhysicsInformed_TunedConstrained(n_sensors=n_sensors).to(device)
        for mod in m.modules():
            if isinstance(mod, S4DKernel):
                for p in [mod.log_dt, mod.log_A_real, mod.A_imag, mod.C]:
                    p._is_ssm = True

        ssm   = [p for p in m.parameters() if getattr(p, "_is_ssm", False)]
        other = [p for p in m.parameters() if not getattr(p, "_is_ssm", False)]
        opt = torch.optim.AdamW([{"params": other, "lr": 5e-4, "weight_decay": 1e-2},
                                 {"params": ssm, "lr": 1e-3, "weight_decay": 0.0}])
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-6)

        ckpt = os.path.join(OUTPUT_DIR, f"tuned_{held_out}.pt")
        best, no_imp = float("inf"), 0
        for ep in range(n_epochs):
            run_epoch_weighted(m, tr_loader, opt)
            va = run_epoch_weighted(m, va_loader)
            sch.step()
            if va < best:
                best, no_imp = va, 0
                torch.save(m.state_dict(), ckpt)
            else:
                no_imp += 1
            if no_imp >= patience:
                break
        m.load_state_dict(torch.load(ckpt, map_location=device))

        ft_opt = torch.optim.AdamW([{"params": other, "lr": 5e-4 * ft_lr_scale, "weight_decay": 1e-2},
                                    {"params": ssm, "lr": 1e-3 * ft_lr_scale, "weight_decay": 0.0}])
        for ep in range(ft_epochs):
            run_epoch_weighted(m, ft_loader, ft_opt)

        m.eval(); P, T = [], []
        with torch.no_grad():
            for cgm, sensor, y, phys in ev_loader:
                pred = m(cgm.to(device), sensor.to(device), phys.to(device))
                P.append(pred.cpu().numpy()); T.append(y.numpy())
        P, T = np.concatenate(P), np.concatenate(T)
        gm, gs = eval_recs[0]["glucose_mean"], eval_recs[0]["glucose_std"]
        P_real, T_real = P * gs + gm, T * gs + gm
        all_preds.append(P_real); all_trues.append(T_real)
        maes = [mean_absolute_error(T_real[:, i], P_real[:, i]) for i in range(3)]
        print(f"  {held_out:16s} | MAE 15/30/60: {maes[0]:.3f}/{maes[1]:.3f}/{maes[2]:.3f}")

    return np.concatenate(all_preds), np.concatenate(all_trues)


print("Running LOSOCV — Tuned Constrained Hybrid...")
P_tuned, T_tuned = run_losocv_tuned()

print("\n" + "="*60)
print("FINAL RESULT — Tuned Constrained Hybrid")
print("="*60)
for i, h in enumerate(["t+15", "t+30", "t+60"]):
    m = mard(T_tuned[:, i], P_tuned[:, i])
    zones = clarke_zone(T_tuned[:, i], P_tuned[:, i])
    ab = (zones == "A").mean()*100 + (zones == "B").mean()*100
    print(f"{h}: MARD = {m:.2f}%   Zone A+B = {ab:.1f}%")

Running LOSOCV — Tuned Constrained Hybrid...
  diabetes_001     | MAE 15/30/60: 0.269/0.707/1.558
  diabetes_002     | MAE 15/30/60: 0.304/0.845/2.027


KeyboardInterrupt: 

In [ ]:
# ── COMPREHENSIVE EVALUATION — Confusion Matrix + Accuracy + Clarke Zones (all horizons) ──
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

def glucose_category(g_mmol):
    """Standard clinical glucose categories (mmol/L): Hypo <3.9, Normal 3.9-10.0, Hyper >10.0"""
    cat = np.full(g_mmol.shape, "Normal", dtype=object)
    cat[g_mmol < 3.9] = "Hypoglycemia"
    cat[g_mmol > 10.0] = "Hyperglycemia"
    return cat

labels_order = ["Hypoglycemia", "Normal", "Hyperglycemia"]
horizons_labels = ["t+15", "t+30", "t+60"]

# ── 1. Confusion Matrices (glucose category) — all 3 horizons side by side ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
category_results = {}

for i, h in enumerate(horizons_labels):
    true_cat = glucose_category(T_tuned[:, i])
    pred_cat = glucose_category(P_tuned[:, i])

    cm = confusion_matrix(true_cat, pred_cat, labels=labels_order)
    cm_pct = np.divide(cm.astype(float), cm.sum(axis=1, keepdims=True),
                       out=np.zeros_like(cm, dtype=float), where=cm.sum(axis=1, keepdims=True)!=0) * 100

    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues",
               xticklabels=labels_order, yticklabels=labels_order, ax=axes[i],
               cbar=(i==2), vmin=0, vmax=100)
    axes[i].set_title(f"{h} — Confusion Matrix (row %)")
    axes[i].set_xlabel("Predicted Category")
    axes[i].set_ylabel("Actual Category")

    category_results[h] = {"true": true_cat, "pred": pred_cat, "cm": cm}

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), dpi=150, bbox_inches='tight')
plt.show()

# ── 2. Classification reports (precision/recall/F1 per category) ──
print("="*70)
print("CLASSIFICATION REPORTS — Glucose Category Prediction")
print("="*70)
for h in horizons_labels:
    print(f"\n--- {h} ---")
    print(classification_report(category_results[h]["true"], category_results[h]["pred"],
                                labels=labels_order, zero_division=0))

# ── 3. Overall category accuracy summary table ──
print("="*70)
print("CATEGORY ACCURACY SUMMARY")
print("="*70)
print(f"{'Horizon':<10}{'Overall Acc':>14}{'Hypo Recall':>14}{'Hyper Recall':>14}")
for h in horizons_labels:
    true_cat, pred_cat = category_results[h]["true"], category_results[h]["pred"]
    overall_acc = accuracy_score(true_cat, pred_cat) * 100
    # recall = of all TRUE hypo/hyper cases, how many did we correctly catch
    hypo_mask = true_cat == "Hypoglycemia"
    hyper_mask = true_cat == "Hyperglycemia"
    hypo_recall = (pred_cat[hypo_mask] == "Hypoglycemia").mean()*100 if hypo_mask.sum() > 0 else float('nan')
    hyper_recall = (pred_cat[hyper_mask] == "Hyperglycemia").mean()*100 if hyper_mask.sum() > 0 else float('nan')
    print(f"{h:<10}{overall_acc:>13.1f}%{hypo_recall:>13.1f}%{hyper_recall:>13.1f}%")

# ── 4. Clarke Error Grid Zone breakdown (regression-level safety, not just category) ──
def clarke_zone(ref, pred):
    ref, pred = ref * 18.0182, pred * 18.0182
    n = len(ref)
    zones = np.full(n, "E", dtype="<U1")
    zone_a = (np.abs(pred - ref) <= 0.2 * ref) | ((ref < 70) & (pred < 70))
    zone_e = ((ref >= 180) & (pred <= 70)) | ((ref <= 70) & (pred >= 180))
    zone_c = ((ref >= 70) & (ref <= 290) & (pred >= ref + 110)) | \
             ((ref >= 130) & (ref <= 180) & (pred <= (7/5) * ref - 182))
    zone_d = ((ref > 240) & (pred >= 70) & (pred <= 180)) | \
             ((ref < 70) & (pred >= 70) & (pred <= 180))
    zone_b = ~zone_a & ~zone_e & ~zone_c & ~zone_d
    zones[zone_e] = "E"; zones[zone_c] = "C"; zones[zone_d] = "D"; zones[zone_b] = "B"; zones[zone_a] = "A"
    return zones

def mard(y_true, y_pred):
    return float(np.mean(np.abs(y_pred - y_true) / y_true) * 100)

print("\n" + "="*70)
print("CLARKE ERROR GRID ZONE DISTRIBUTION (%)")
print("="*70)
print(f"{'Horizon':<10}{'Zone A':>9}{'Zone B':>9}{'Zone C':>9}{'Zone D':>9}{'Zone E':>9}{'A+B':>9}")
for i, h in enumerate(horizons_labels):
    zones = clarke_zone(T_tuned[:, i], P_tuned[:, i])
    zp = {z: (zones == z).mean()*100 for z in "ABCDE"}
    ab = zp["A"] + zp["B"]
    print(f"{h:<10}{zp['A']:>8.1f}%{zp['B']:>8.1f}%{zp['C']:>8.1f}%{zp['D']:>8.1f}%{zp['E']:>8.1f}%{ab:>8.1f}%")

# ── 5. Master summary table — everything in one place ──
print("\n" + "="*70)
print("MASTER SUMMARY — Final Tuned Constrained Hybrid Model")
print("="*70)
print(f"{'Horizon':<10}{'MARD':>10}{'Zone A+B':>12}{'Category Acc':>15}{'Hypo Recall':>14}")
for i, h in enumerate(horizons_labels):
    m = mard(T_tuned[:, i], P_tuned[:, i])
    zones = clarke_zone(T_tuned[:, i], P_tuned[:, i])
    ab = (zones == "A").mean()*100 + (zones == "B").mean()*100
    true_cat, pred_cat = category_results[h]["true"], category_results[h]["pred"]
    acc = accuracy_score(true_cat, pred_cat) * 100
    hypo_mask = true_cat == "Hypoglycemia"
    hypo_recall = (pred_cat[hypo_mask] == "Hypoglycemia").mean()*100 if hypo_mask.sum() > 0 else float('nan')
    print(f"{h:<10}{m:>9.2f}%{ab:>11.1f}%{acc:>14.1f}%{hypo_recall:>13.1f}%")

In [ ]:
# ── ROC Curves — one-vs-rest for each glucose category, using predicted value as score ──
from sklearn.metrics import roc_curve, auc

def make_roc_plots(T, P, title_prefix=""):
    horizons_labels = ["t+15", "t+30", "t+60"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for i, h in enumerate(horizons_labels):
        true_vals = T[:, i]
        pred_vals = P[:, i]

        # Binary labels: is this sample actually Hypoglycemia? (< 3.9 mmol/L)
        y_true_hypo = (true_vals < 3.9).astype(int)
        # Score: LOWER predicted glucose = more confident it's hypo, so we invert for ROC convention
        y_score_hypo = -pred_vals   # higher score = more "hypo-like"

        y_true_hyper = (true_vals > 10.0).astype(int)
        y_score_hyper = pred_vals   # higher predicted glucose = more "hyper-like"

        ax = axes[i]
        for label, y_true, y_score, color in [
            ("Hypoglycemia", y_true_hypo, y_score_hypo, "crimson"),
            ("Hyperglycemia", y_true_hyper, y_score_hyper, "darkorange"),
        ]:
            if y_true.sum() == 0 or y_true.sum() == len(y_true):
                continue  # skip if no positive/negative examples
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f"{label} (AUC={roc_auc:.3f})", color=color, lw=2)

        ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label="Random (AUC=0.5)")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"{title_prefix}{h} — ROC Curve")
        ax.legend(loc="lower right", fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curves.png"), dpi=150, bbox_inches='tight')
    plt.show()


print("Generating ROC curves for Tuned Constrained Hybrid model...")
make_roc_plots(T_tuned, P_tuned, title_prefix="Final Model — ")

# Print AUC summary table
print("\n" + "="*60)
print("AUC SUMMARY")
print("="*60)
print(f"{'Horizon':<10}{'Hypo AUC':>14}{'Hyper AUC':>14}")
for i, h in enumerate(["t+15", "t+30", "t+60"]):
    true_vals, pred_vals = T_tuned[:, i], P_tuned[:, i]
    y_true_hypo = (true_vals < 3.9).astype(int)
    y_true_hyper = (true_vals > 10.0).astype(int)
    auc_hypo = auc(*roc_curve(y_true_hypo, -pred_vals)[:2]) if 0 < y_true_hypo.sum() < len(y_true_hypo) else float('nan')
    auc_hyper = auc(*roc_curve(y_true_hyper, pred_vals)[:2]) if 0 < y_true_hyper.sum() < len(y_true_hyper) else float('nan')
    print(f"{h:<10}{auc_hypo:>13.3f}{auc_hyper:>13.3f}")

In [ ]:
# ── Threshold selection on TRAINING folds only, then frozen and applied to held-out subject ──
from sklearn.metrics import roc_curve, precision_score, recall_score

def select_threshold_on_train(train_true, train_pred, target_recall=0.80):
    """Select threshold using ONLY training-fold data (never touches held-out subject).
    Picks the threshold achieving closest-to-target recall on training predictions."""
    y_true = (train_true < 3.9).astype(int)
    y_score = -train_pred   # higher score = more hypo-like
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    # Find threshold closest to target recall
    idx = np.argmin(np.abs(tpr - target_recall))
    tau = -thresholds[idx]   # convert back to glucose-value threshold
    return tau, tpr[idx], fpr[idx]


def apply_consecutive_rule(pred_series, tau, n_required=2, window=3):
    """Alarm fires only if >= n_required of the last `window` predictions are below tau."""
    below = (pred_series < tau).astype(int)
    alarm = np.zeros_like(below)
    for i in range(window - 1, len(below)):
        if below[i - window + 1:i + 1].sum() >= n_required:
            alarm[i] = 1
    return alarm


# ── Proper LOSOCV-consistent evaluation: for EACH held-out subject, tau is chosen
#    using only the OTHER 8 subjects' predictions from this same model (train-fold proxy) ──
print("="*80)
print("THRESHOLD SWEEP — selected on training folds, applied unchanged to held-out subject")
print("="*80)

results_per_horizon = {}

for i, h in enumerate(["t+15", "t+30", "t+60"]):
    all_recalls, all_precisions, all_taus, all_fa_per_day = [], [], [], []

    # Build per-subject prediction arrays (needed to hold one out for tau selection)
    # NOTE: assumes T_tuned/P_tuned were built subject-by-subject in the same order as subj_ids
    # If you have per-subject splits stored separately, use those instead of this global array.
    n_total = len(T_tuned)
    fold_size = n_total // len(subj_ids)  # approximate split points

    for held_out_idx, held_out in enumerate(subj_ids):
        start = held_out_idx * fold_size
        end = start + fold_size if held_out_idx < len(subj_ids) - 1 else n_total

        # "test" = this subject's held-out predictions
        test_true = T_tuned[start:end, i]
        test_pred = P_tuned[start:end, i]

        # "train" = all OTHER subjects' predictions (proxy for training-fold selection)
        train_true = np.concatenate([T_tuned[:start, i], T_tuned[end:, i]])
        train_pred = np.concatenate([P_tuned[:start, i], P_tuned[end:, i]])

        if (train_true < 3.9).sum() == 0 or (test_true < 3.9).sum() == 0:
            continue  # skip if no hypo examples to calibrate/evaluate on

        tau, train_recall, train_fpr = select_threshold_on_train(train_true, train_pred, target_recall=0.80)

        # Apply frozen tau to held-out subject — NO tuning on this data
        y_true_test = (test_true < 3.9).astype(int)
        y_pred_test = (test_pred < tau).astype(int)

        if y_pred_test.sum() == 0:
            recall, precision = 0.0, float('nan')
        else:
            recall = recall_score(y_true_test, y_pred_test, zero_division=0)
            precision = precision_score(y_true_test, y_pred_test, zero_division=0)

        # False alarms per day: assume 5-min sampling -> 288 samples/day
        n_false_alarms = ((y_pred_test == 1) & (y_true_test == 0)).sum()
        n_days = len(test_true) / 288
        fa_per_day = n_false_alarms / n_days if n_days > 0 else float('nan')

        all_recalls.append(recall); all_precisions.append(precision)
        all_taus.append(tau); all_fa_per_day.append(fa_per_day)

    results_per_horizon[h] = {
        "tau_mean": np.nanmean(all_taus), "tau_std": np.nanstd(all_taus),
        "recall_mean": np.nanmean(all_recalls), "recall_std": np.nanstd(all_recalls),
        "precision_mean": np.nanmean(all_precisions),
        "fa_per_day_mean": np.nanmean(all_fa_per_day)
    }

    print(f"\n{h}:")
    print(f"  τ (threshold)     : {results_per_horizon[h]['tau_mean']:.2f} ± {results_per_horizon[h]['tau_std']:.2f} mmol/L")
    print(f"  Recall (held-out) : {results_per_horizon[h]['recall_mean']*100:.1f}% ± {results_per_horizon[h]['recall_std']*100:.1f}%")
    print(f"  Precision         : {results_per_horizon[h]['precision_mean']*100:.1f}%")
    print(f"  False alarms/day  : {results_per_horizon[h]['fa_per_day_mean']:.2f}")

In [ ]:
print("\n" + "="*80)
print("EFFECT OF 2-OF-3 CONSECUTIVE-SAMPLE ALARM RULE (t+60)")
print("="*80)

h_idx = 2  # t+60
tau_final = results_per_horizon["t+60"]["tau_mean"]

# Apply single-sample vs consecutive-rule alarms on the full prediction series
single_alarm = (P_tuned[:, h_idx] < tau_final).astype(int)
consec_alarm = apply_consecutive_rule(P_tuned[:, h_idx], tau_final, n_required=2, window=3)

y_true_full = (T_tuned[:, h_idx] < 3.9).astype(int)

for label, alarm in [("Single-sample", single_alarm), ("2-of-3 consecutive", consec_alarm)]:
    recall = recall_score(y_true_full, alarm, zero_division=0)
    precision = precision_score(y_true_full, alarm, zero_division=0)
    n_days = len(y_true_full) / 288
    fa_per_day = ((alarm == 1) & (y_true_full == 0)).sum() / n_days
    print(f"{label:<20}: Recall={recall*100:.1f}%  Precision={precision*100:.1f}%  FA/day={fa_per_day:.2f}")

In [ ]:
# ── KEY FIGURE — τ growth vs a_g decay (the cleanest evidence of shrinkage) ──

horizons_min = [15, 30, 60]
tau_values = [3.79, 4.19, 5.69]       # from your threshold sweep results
a_g_values = [0.988, 0.958, 0.863]    # from your empirical-v3 formula coefficients

fig, ax1 = plt.subplots(figsize=(8, 5.5))

color1 = 'crimson'
ax1.set_xlabel('Forecast Horizon (minutes)', fontsize=12)
ax1.set_ylabel('Optimal Hypo Threshold τ (mmol/L)', color=color1, fontsize=12)
ax1.plot(horizons_min, tau_values, 'o-', color=color1, linewidth=2.5, markersize=9, label='τ (threshold)')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(horizons_min)

ax2 = ax1.twinx()
color2 = 'steelblue'
ax2.set_ylabel('a_g (weight on current glucose)', color=color2, fontsize=12)
ax2.plot(horizons_min, a_g_values, 's-', color=color2, linewidth=2.5, markersize=9, label='a_g (shrinkage coefficient)')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Threshold Drift Mirrors Model Shrinkage\n(τ rises as a_g falls with forecast horizon)', fontsize=13)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=10)

fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "tau_vs_shrinkage.png"), dpi=150, bbox_inches='tight')
plt.show()

# Print correlation as numeric backup
from scipy.stats import pearsonr
corr, pval = pearsonr(a_g_values, tau_values)
print(f"Correlation between a_g and τ: r = {corr:.3f} (p = {pval:.3f})")
print(f"As a_g decreases (more shrinkage), τ increases (more compensation needed) — inverse relationship confirmed.")

In [ ]:
def apply_consecutive_rule_fixed(pred_series, tau, n_required=2, window=3):
    """
    CORRECTED: sample i is flagged ONLY IF:
      (a) sample i itself crosses tau, AND
      (b) at least n_required of the last `window` samples (including i) cross tau
    This guarantees alarms are a SUBSET of single-sample alarms — filter can only reduce, never add.
    """
    below = (pred_series < tau).astype(int)
    alarm = np.zeros_like(below)
    for i in range(window - 1, len(below)):
        if below[i] == 1 and below[i - window + 1:i + 1].sum() >= n_required:
            alarm[i] = 1
    return alarm


# ── Re-run comparison with the corrected rule ──
h_idx = 2  # t+60
tau_final = results_per_horizon["t+60"]["tau_mean"]

single_alarm = (P_tuned[:, h_idx] < tau_final).astype(int)
consec_alarm_fixed = apply_consecutive_rule_fixed(P_tuned[:, h_idx], tau_final, n_required=2, window=3)

y_true_full = (T_tuned[:, h_idx] < 3.9).astype(int)

print("="*70)
print("CORRECTED — 2-of-3 rule is now a strict subset of single-sample alarms")
print("="*70)
for label, alarm in [("Single-sample", single_alarm), ("2-of-3 (fixed)", consec_alarm_fixed)]:
    recall = recall_score(y_true_full, alarm, zero_division=0)
    precision = precision_score(y_true_full, alarm, zero_division=0)
    n_days = len(y_true_full) / 288
    fa_per_day = ((alarm == 1) & (y_true_full == 0)).sum() / n_days
    print(f"{label:<20}: Recall={recall*100:.1f}%  Precision={precision*100:.1f}%  FA/day={fa_per_day:.2f}")

# Sanity check: consec alarms must be subset of single alarms
assert (consec_alarm_fixed <= single_alarm).all(), "Bug still present — consec alarm not a subset!"
print("\nSanity check passed: consecutive-rule alarms are a strict subset of single-sample alarms.")

In [ ]:
horizons_min = [30, 60]   # t+15 excluded — shrinkage effect not present there (τ=3.79 < 3.9)
tau_values = [4.19, 5.69]
a_g_values = [0.958, 0.863]

fig, ax1 = plt.subplots(figsize=(8, 5.5))

color1 = 'crimson'
ax1.set_xlabel('Forecast Horizon (minutes)', fontsize=12)
ax1.set_ylabel('Optimal Hypo Threshold τ (mmol/L)', color=color1, fontsize=12)
ax1.plot(horizons_min, tau_values, 'o-', color=color1, linewidth=2.5, markersize=9, label='τ (threshold)')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(horizons_min)

ax2 = ax1.twinx()
color2 = 'steelblue'
ax2.set_ylabel('a_g (weight on current glucose)', color=color2, fontsize=12)
ax2.plot(horizons_min, a_g_values, 's-', color=color2, linewidth=2.5, markersize=9, label='a_g (shrinkage coefficient)')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Threshold Drift Mirrors Model Shrinkage (t+30 to t+60)\n'
         'Illustrative mechanism, not a statistical test', fontsize=12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=10)

fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "tau_vs_shrinkage_corrected.png"), dpi=150, bbox_inches='tight')
plt.show()

print("Note: t+15 excluded from this illustration — τ=3.79 is BELOW the clinical 3.9 cutoff,")
print("indicating no shrinkage-compensation is needed at that horizon. The mechanism is")
print("horizon-dependent, emerging at t+30/t+60, not universal across all horizons.")

In [ ]:
print("="*80)
print("FINAL CORRECTED SUMMARY — with confidence intervals and honest deployability")
print("="*80)
print(f"{'Horizon':<10}{'τ':>8}{'Recall':>18}{'Precision':>12}{'FA/day':>10}{'Deployable?':>14}")
data = [
    ("t+15", 3.79, 70.2, 21.8, 81.3, 1.98, "Yes"),
    ("t+30", 4.19, 73.6, 19.9, 55.7, 9.28, "Borderline"),
    ("t+60", 5.69, 78.1, 18.3, 27.4, 36.99, "No"),
]
for h, tau, rec, rec_std, prec, fa, deploy in data:
    print(f"{h:<10}{tau:>7.2f}{rec:>10.1f}% ± {rec_std:>4.1f}%{prec:>11.1f}%{fa:>9.2f}{deploy:>15}")

In [ ]:

# ── UNIFIED per-subject-fold pipeline — single source of truth for all downstream metrics ──
from sklearn.metrics import recall_score, precision_score

def unified_hypo_pipeline(target_recall=0.80):
    """
    For each held-out subject: select tau on the OTHER 8 subjects' pooled predictions,
    apply unchanged to held-out subject. Store per-sample alarms (single + corrected-2of3)
    AND per-subject arrays, so every downstream metric (recall, precision, FA/day, episodes)
    comes from this exact same set of alarms — no more mismatched pooling.
    """
    per_subject_data = []  # list of dicts per subject: true, pred, single_alarm, consec_alarm, tau

    for h_idx, h_label in enumerate(["t+15", "t+30", "t+60"]):
        n_total = len(T_tuned)
        fold_size = n_total // len(subj_ids)

        for held_out_idx, held_out in enumerate(subj_ids):
            start = held_out_idx * fold_size
            end = start + fold_size if held_out_idx < len(subj_ids) - 1 else n_total

            test_true = T_tuned[start:end, h_idx]
            test_pred = P_tuned[start:end, h_idx]
            train_true = np.concatenate([T_tuned[:start, h_idx], T_tuned[end:, h_idx]])
            train_pred = np.concatenate([T_tuned[:start, h_idx], T_tuned[end:, h_idx]]) if False else \
                        np.concatenate([P_tuned[:start, h_idx], P_tuned[end:, h_idx]])

            if (train_true < 3.9).sum() == 0 or (test_true < 3.9).sum() == 0:
                continue

            tau, _, _ = select_threshold_on_train(train_true, train_pred, target_recall=target_recall)

            single_alarm = (test_pred < tau).astype(int)
            consec_alarm = apply_consecutive_rule_fixed(test_pred, tau, n_required=2, window=3)

            per_subject_data.append({
                "horizon": h_label, "subject": held_out, "tau": tau,
                "true": test_true, "pred": test_pred,
                "single_alarm": single_alarm, "consec_alarm": consec_alarm
            })

    return per_subject_data

pipeline_data = unified_hypo_pipeline()
print(f"Unified pipeline built: {len(pipeline_data)} subject-horizon folds")

CELL B — Micro-pooled metrics (final, reconciled numbers) — reported consistently everywhere

In [ ]:
print("="*90)
print("RECONCILED FINAL TABLE — single pipeline, single set of numbers")
print("="*90)
print(f"{'Horizon':<10}{'τ(mean)':>9}{'Recall':>10}{'Precision':>11}{'FA/day':>9}   | 2-of-3: {'Recall':>8}{'Precision':>11}{'FA/day':>9}")

for h_label in ["t+15", "t+30", "t+60"]:
    rows = [d for d in pipeline_data if d["horizon"] == h_label]
    all_true = np.concatenate([d["true"] for d in rows])
    all_single = np.concatenate([d["single_alarm"] for d in rows])
    all_consec = np.concatenate([d["consec_alarm"] for d in rows])
    y_true = (all_true < 3.9).astype(int)
    tau_mean = np.mean([d["tau"] for d in rows])
    n_days = len(all_true) / 288

    rec_s = recall_score(y_true, all_single, zero_division=0)*100
    prec_s = precision_score(y_true, all_single, zero_division=0)*100
    fa_s = ((all_single==1)&(y_true==0)).sum()/n_days

    rec_c = recall_score(y_true, all_consec, zero_division=0)*100
    prec_c = precision_score(y_true, all_consec, zero_division=0)*100
    fa_c = ((all_consec==1)&(y_true==0)).sum()/n_days

    print(f"{h_label:<10}{tau_mean:>8.2f}{rec_s:>9.1f}%{prec_s:>10.1f}%{fa_s:>8.2f}   |{rec_c:>16.1f}%{prec_c:>10.1f}%{fa_c:>8.2f}")

CELL C — Item 4 (PRIORITY): Episode-Level Metrics

In [ ]:
# ── Episode-level detection — per subject, per horizon ──
from scipy.stats import norm

def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = successes / n
    denom = 1 + z**2/n
    center = (p + z**2/(2*n)) / denom
    margin = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return (center - margin, center + margin)


def extract_episodes(is_hypo, min_gap_samples=3, min_duration_samples=1):
    """min_gap_samples=3 -> merge episodes <15min apart (5-min sampling).
    min_duration_samples=1 -> drop single-sample blips (set to 2 to require >=10min)."""
    if is_hypo.sum() == 0:
        return []
    episode_id = (is_hypo.astype(int) != np.roll(is_hypo.astype(int), 1)).cumsum()
    episode_id[0] = 1 if is_hypo[0] else 0

    raw_episodes = []
    for eid in np.unique(episode_id):
        mask = (episode_id == eid) & is_hypo
        idxs = np.where(mask)[0]
        if len(idxs) > 0:
            raw_episodes.append((idxs[0], idxs[-1]))

    # merge episodes closer than min_gap_samples
    merged = []
    for onset, offset in raw_episodes:
        if merged and onset - merged[-1][1] <= min_gap_samples:
            merged[-1] = (merged[-1][0], offset)
        else:
            merged.append((onset, offset))

    # drop too-short episodes
    merged = [(o, f) for o, f in merged if (f - o + 1) >= min_duration_samples]
    return merged


def event_level_metrics(rows, lookback_samples=12):
    """For each episode, check if any alarm fired in [onset-lookback, onset] window."""
    all_detected, all_lead_times, n_episodes = [], [], 0

    for d in rows:
        true, single_alarm = d["true"], d["single_alarm"]
        is_hypo = true < 3.9
        episodes = extract_episodes(is_hypo, min_gap_samples=3, min_duration_samples=1)
        n_episodes += len(episodes)

        for onset, offset in episodes:
            window_start = max(0, onset - lookback_samples)
            alarm_window = single_alarm[window_start:onset+1]
            fired_idxs = np.where(alarm_window == 1)[0]
            if len(fired_idxs) > 0:
                all_detected.append(1)
                first_alarm_offset = fired_idxs[0]
                lead_time_samples = onset - (window_start + first_alarm_offset)
                all_lead_times.append(lead_time_samples * 5)  # convert to minutes
            else:
                all_detected.append(0)

    return n_episodes, all_detected, all_lead_times


print("="*90)
print("EPISODE-LEVEL METRICS")
print("="*90)
for h_label in ["t+15", "t+30", "t+60"]:
    rows = [d for d in pipeline_data if d["horizon"] == h_label]
    n_episodes, detected, lead_times = event_level_metrics(rows)

    n_detected = sum(detected)
    event_recall = n_detected / n_episodes * 100 if n_episodes > 0 else float('nan')
    ci_low, ci_high = wilson_ci(n_detected, n_episodes)

    if lead_times:
        median_lead = np.median(lead_times)
        q1, q3 = np.percentile(lead_times, [25, 75])
    else:
        median_lead, q1, q3 = float('nan'), float('nan'), float('nan')

    print(f"\n{h_label}:")
    print(f"  Number of episodes    : {n_episodes}")
    print(f"  Event recall          : {event_recall:.1f}%  (Wilson 95% CI: {ci_low*100:.1f}%-{ci_high*100:.1f}%)")
    print(f"  Median lead time      : {median_lead:.1f} min  (IQR: {q1:.1f}-{q3:.1f} min)")
    print(f"  [compare to sample-level recall for context]")

CELL D — Item 1: k1/k2 Diagnostics

In [ ]:
# ── Diagnose why k1(carbs)/k2(units) were exactly 0 ──
merged_check = subject_merged[subj_ids[0]]
carbs_arr = merged_check["recent_carbs"].values
units_arr = merged_check["recent_units"].values

print("="*60)
print(f"DIAGNOSTIC — {subj_ids[0]}")
print("="*60)
print(f"Total carbs sum   : {carbs_arr.sum():.2f}")
print(f"Total units sum   : {units_arr.sum():.2f}")
print(f"Non-zero carb rows: {(carbs_arr != 0).sum()} / {len(carbs_arr)}")
print(f"Non-zero unit rows: {(units_arr != 0).sum()} / {len(units_arr)}")
print(f"Max carbs value   : {carbs_arr.max():.2f}")
print(f"Max units value   : {units_arr.max():.2f}")

# Check across ALL subjects
print("\n" + "="*60)
print("ACROSS ALL SUBJECTS")
print("="*60)
for s in subj_ids:
    m = subject_merged[s]
    c, u = m["recent_carbs"].values, m["recent_units"].values
    print(f"{s:16s}: nonzero_carbs={  (c!=0).sum():4d}  nonzero_units={(u!=0).sum():4d}  "
          f"max_carb={c.max():6.1f}  max_unit={u.max():5.1f}")

In [ ]:
# Quick check — are food.csv timestamps even parseable / in range?
subject_dir = os.path.dirname(all_subjects[0]["glucose_csv"])
food_path = os.path.join(subject_dir, "food.csv")
food_df = pd.read_csv(food_path)
print(food_df.head())
print(food_df.dtypes)
food_df["datetime"] = pd.to_datetime(food_df["date"] + " " + food_df["time"], errors="coerce")
print("Parsed NaT count:", food_df["datetime"].isna().sum(), "/", len(food_df))
print("Date range:", food_df["datetime"].min(), "to", food_df["datetime"].max())

gdf = load_glucose(all_subjects[0]["glucose_csv"])
print("CGM date range:", gdf["datetime"].min(), "to", gdf["datetime"].max())

CELL E — Item 2: Persistence Baseline (+ hypo threshold sweep)

In [ ]:
# ── Persistence baseline — g(t+h) = g(t), same folds, same metrics ──
def evaluate_persistence():
    all_P, all_T = {h: [] for h in HORIZONS}, {h: [] for h in HORIZONS}
    for held_out in subj_ids:
        merged = subject_merged[held_out]
        g = merged["glucose"].values.astype(np.float64)
        max_h = max(HORIZONS)
        for h in HORIZONS:
            for i in range(WINDOW_SIZE, len(g) - max_h):
                all_P[h].append(g[i])   # persistence: predict current value
                all_T[h].append(g[i+h])
    return all_P, all_T

P_persist, T_persist = evaluate_persistence()

print("="*70)
print("PERSISTENCE BASELINE vs Physics-v3")
print("="*70)
for h, label in zip(HORIZONS, ["t+15", "t+30", "t+60"]):
    p, t = np.array(P_persist[h]), np.array(T_persist[h])
    m_pers = mard(t, p)
    print(f"{label}: Persistence MARD = {m_pers:.2f}%   (vs Physics-v3: check earlier results)")

# Hypo threshold sweep on persistence, t+60
h60 = HORIZONS[2]
p60, t60 = np.array(P_persist[h60]), np.array(T_persist[h60])
y_true_hypo = (t60 < 3.9).astype(int)
if y_true_hypo.sum() > 0:
    tau_persist, recall_persist, fpr_persist = select_threshold_on_train(t60, p60, target_recall=0.80)
    print(f"\nPersistence hypo detector (t+60): τ={tau_persist:.2f}, recall≈{recall_persist*100:.1f}%")
    print("Compare this directly against Physics-v3's τ=5.69, recall=78.1% (from unified pipeline)")

CELL F — Item 3: S4D Verdict (correction magnitude + paired Wilcoxon)

In [ ]:
def evaluate_physics_v3_standalone():
    all_P, all_T = {h: [] for h in HORIZONS}, {h: [] for h in HORIZONS}
    for held_out in subj_ids:
        train_subjs = [s for s in subj_ids if s != held_out]
        train_merged = [subject_merged[s] for s in train_subjs]
        merged = subject_merged[held_out]
        g = merged["glucose"].values.astype(np.float64)
        max_h = max(HORIZONS)
        for h in HORIZONS:
            coefs = fit_empirical_formula_v3(train_merged, h)
            for i in range(12, len(g) - max_h):
                g_hist = g[i-12:i+1]
                all_P[h].append(empirical_predict_v3(g_hist, h, coefs))
                all_T[h].append(g[i + h])
    return all_P, all_T

P_v3, T_v3 = evaluate_physics_v3_standalone()
print("P_v3 ready — now re-run the Wilcoxon cell.")

In [ ]:
from scipy.stats import wilcoxon

# ── Correction magnitude: how much does the NN actually change the physics estimate? ──
# Requires P_v3 (physics-only) and P_tuned (hybrid) aligned per-subject — recompute if needed
print("="*60)
print("S4D CORRECTION MAGNITUDE (per horizon)")
print("="*60)
# NOTE: this needs physics_pred stored per-sample in eval_recs during the hybrid LOSOCV run.
# If not stored, approximate using physics-only P_v3 vs hybrid P_tuned — lengths may not align
# perfectly across the two separate LOSOCV runs; recompute both under the SAME windowing if needed.

for i, h in enumerate(["t+15", "t+30", "t+60"]):
    p_v3 = np.array(P_v3[HORIZONS[i]])
    p_hyb = P_tuned[:len(p_v3), i]
    n = min(len(p_v3), len(p_hyb))
    corr = p_hyb[:n] - p_v3[:n]
    print(f"{h}: mean|correction| = {np.abs(corr).mean():.3f} mmol/L, max|correction| = {np.abs(corr).max():.3f} mmol/L")

# ── Paired Wilcoxon on per-subject MAE (physics vs hybrid), n=9 ──
print("\n" + "="*60)
print("PAIRED WILCOXON — Physics-v3 vs Hybrid (per-subject MAE, t+60)")
print("="*60)
mae_physics_per_subj, mae_hybrid_per_subj = [], []
for held_out_idx, held_out in enumerate(subj_ids):
    n_total = len(T_tuned)
    fold_size = n_total // len(subj_ids)
    start = held_out_idx * fold_size
    end = start + fold_size if held_out_idx < len(subj_ids)-1 else n_total
    mae_hybrid_per_subj.append(mean_absolute_error(T_tuned[start:end, 2], P_tuned[start:end, 2]))

    # match physics-v3 for same subject (assumes same subject order in P_v3/T_v3 — verify!)
    t60_key = HORIZONS[2]
    p_v3_subj = np.array(P_v3[t60_key])[start:end] if end <= len(P_v3[t60_key]) else None
    t_v3_subj = np.array(T_v3[t60_key])[start:end] if end <= len(T_v3[t60_key]) else None
    if p_v3_subj is not None:
        mae_physics_per_subj.append(mean_absolute_error(t_v3_subj, p_v3_subj))

if len(mae_physics_per_subj) == len(mae_hybrid_per_subj) == 9:
    stat, pval = wilcoxon(mae_physics_per_subj, mae_hybrid_per_subj)
    median_diff = np.median(np.array(mae_hybrid_per_subj) - np.array(mae_physics_per_subj))
    print(f"Median paired difference (hybrid - physics): {median_diff:.4f} mmol/L")
    print(f"Wilcoxon p-value: {pval:.3f}")
    print(f"Conclusion: {'No detectable difference (n=9, low power)' if pval > 0.05 else 'Detectable difference'}")
else:
    print("WARNING: subject alignment between physics-only and hybrid runs unclear — verify pairing before reporting.")

In [ ]:
def event_level_metrics_fixed(rows, h_steps, lookback_samples=12):
    """h_steps = HORIZONS value (3, 6, or 12) — the offset between prediction-index i
    and the actual event time (i+h_steps) that the model is predicting for."""
    all_detected, all_lead_times, n_episodes = [], [], 0

    for d in rows:
        true, single_alarm = d["true"], d["single_alarm"]
        is_hypo = true < 3.9
        episodes = extract_episodes(is_hypo, min_gap_samples=3, min_duration_samples=1)
        n_episodes += len(episodes)

        for onset, offset in episodes:
            # onset is in "target index" space; the alarm raised at prediction-index i
            # fires at REAL time i*5min, warning about event at REAL time (i+h_steps)*5min.
            # An alarm at index i is "before" the event onset (real time onset+h_steps) if:
            #   i <= onset   (since alarm made at i predicts for i+h_steps, and if i<=onset,
            #                 the alarm's target i+h_steps could be <= onset+h_steps, i.e. same/earlier event)
            window_start = max(0, onset - lookback_samples)
            alarm_window = single_alarm[window_start:onset+1]
            fired_idxs = np.where(alarm_window == 1)[0]
            if len(fired_idxs) > 0:
                all_detected.append(1)
                first_alarm_idx = window_start + fired_idxs[0]
                # Real lead time: event real-time = (onset+h_steps)*5; alarm real-time = first_alarm_idx*5
                lead_time_min = (onset + h_steps - first_alarm_idx) * 5
                all_lead_times.append(lead_time_min)
            else:
                all_detected.append(0)

    return n_episodes, all_detected, all_lead_times


print("="*90)
print("EPISODE-LEVEL METRICS — CORRECTED (horizon-offset aware)")
print("="*90)
for h_label, h_steps in zip(["t+15", "t+30", "t+60"], HORIZONS):
    rows = [d for d in pipeline_data if d["horizon"] == h_label]
    n_episodes, detected, lead_times = event_level_metrics_fixed(rows, h_steps)
    n_detected = sum(detected)
    event_recall = n_detected / n_episodes * 100 if n_episodes > 0 else float('nan')
    ci_low, ci_high = wilson_ci(n_detected, n_episodes)
    median_lead = np.median(lead_times) if lead_times else float('nan')
    q1, q3 = np.percentile(lead_times, [25, 75]) if lead_times else (float('nan'), float('nan'))
    print(f"\n{h_label}: episodes={n_episodes}  event_recall={event_recall:.1f}% "
          f"(CI {ci_low*100:.1f}-{ci_high*100:.1f}%)  median_lead={median_lead:.1f}min (IQR {q1:.1f}-{q3:.1f})")

In [ ]:
# Sanity check: how many episodes have ANY alarm anywhere nearby (not just in strict window)?
h_label, h_steps = "t+15", 3
rows = [d for d in pipeline_data if d["horizon"] == h_label]

for d in rows[:2]:  # inspect first 2 subjects
    true, single_alarm = d["true"], d["single_alarm"]
    is_hypo = true < 3.9
    episodes = extract_episodes(is_hypo, min_gap_samples=3, min_duration_samples=1)
    print(f"\nSubject: {d['subject']}, {len(episodes)} episodes")
    for onset, offset in episodes[:3]:
        print(f"  Episode at index {onset}-{offset} (true value: {true[onset]:.2f})")
        print(f"  Alarm status in [-20, +5] window: {single_alarm[max(0,onset-20):onset+5]}")

In [ ]:
h_label, h_steps = "t+15", 3
rows = [d for d in pipeline_data if d["horizon"] == h_label]

miss_count, hit_count = 0, 0
for d in rows:
    true, single_alarm = d["true"], d["single_alarm"]
    is_hypo = true < 3.9
    episodes = extract_episodes(is_hypo, min_gap_samples=3, min_duration_samples=1)
    for onset, offset in episodes:
        window_start = max(0, onset - 12)
        alarm_window = single_alarm[window_start:onset+1]
        if (alarm_window == 1).any():
            hit_count += 1
        else:
            miss_count += 1
            print(f"MISS: subject={d['subject']}, onset={onset}, true={true[onset]:.2f}, "
                  f"alarm_window={single_alarm[window_start:onset+1]}")

print(f"\nTotal: hit={hit_count}, miss={miss_count}, recall={hit_count/(hit_count+miss_count)*100:.1f}%")

In [ ]:
def select_threshold_on_train_fixed(train_true, train_pred, target_recall=0.80, clinical_cutoff=3.9):
    """Same as before, but floors tau at the clinical cutoff — an alarm threshold
    should never be STRICTER than the definition of the event itself."""
    y_true = (train_true < clinical_cutoff).astype(int)
    y_score = -train_pred
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    idx = np.argmin(np.abs(tpr - target_recall))
    tau = -thresholds[idx]
    tau = max(tau, clinical_cutoff)   # NEW: never go below clinical cutoff
    return tau, tpr[idx], fpr[idx]

In [ ]:
for h_label in ["t+15", "t+30", "t+60"]:
    rows = [d for d in pipeline_data if d["horizon"] == h_label]
    taus = [d["tau"] for d in rows]
    print(f"{h_label}: tau range = {min(taus):.2f} to {max(taus):.2f}  (clinical cutoff = 3.9)")

In [ ]:
# ── Fixed threshold selection — floors tau at clinical cutoff ──
def select_threshold_on_train_fixed(train_true, train_pred, target_recall=0.80, clinical_cutoff=3.9):
    y_true = (train_true < clinical_cutoff).astype(int)
    y_score = -train_pred
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    idx = np.argmin(np.abs(tpr - target_recall))
    tau = -thresholds[idx]
    tau = max(tau, clinical_cutoff)   # never go below clinical cutoff
    return tau, tpr[idx], fpr[idx]


# ── Rebuild pipeline data ONLY for t+15 (t+30/t+60 already correct — tau > 3.9) ──
def rebuild_t15_with_fixed_threshold(target_recall=0.80):
    per_subject_data_t15 = []
    h_idx, h_label = 0, "t+15"
    n_total = len(T_tuned)
    fold_size = n_total // len(subj_ids)

    for held_out_idx, held_out in enumerate(subj_ids):
        start = held_out_idx * fold_size
        end = start + fold_size if held_out_idx < len(subj_ids) - 1 else n_total

        test_true = T_tuned[start:end, h_idx]
        test_pred = P_tuned[start:end, h_idx]
        train_true = np.concatenate([T_tuned[:start, h_idx], T_tuned[end:, h_idx]])
        train_pred = np.concatenate([P_tuned[:start, h_idx], P_tuned[end:, h_idx]])

        if (train_true < 3.9).sum() == 0 or (test_true < 3.9).sum() == 0:
            continue

        tau, _, _ = select_threshold_on_train_fixed(train_true, train_pred, target_recall=target_recall)
        single_alarm = (test_pred < tau).astype(int)
        consec_alarm = apply_consecutive_rule_fixed(test_pred, tau, n_required=2, window=3)

        per_subject_data_t15.append({
            "horizon": h_label, "subject": held_out, "tau": tau,
            "true": test_true, "pred": test_pred,
            "single_alarm": single_alarm, "consec_alarm": consec_alarm
        })
    return per_subject_data_t15


t15_fixed_data = rebuild_t15_with_fixed_threshold()
taus_t15 = [d["tau"] for d in t15_fixed_data]
print(f"t+15 (fixed): tau range = {min(taus_t15):.2f} to {max(taus_t15):.2f}")

# ── Re-run episode-level metrics for t+15 with fixed data ──
n_episodes, detected, lead_times = event_level_metrics_fixed(t15_fixed_data, h_steps=3)
n_detected = sum(detected)
event_recall = n_detected / n_episodes * 100 if n_episodes > 0 else float('nan')
ci_low, ci_high = wilson_ci(n_detected, n_episodes)
median_lead = np.median(lead_times) if lead_times else float('nan')
q1, q3 = np.percentile(lead_times, [25, 75]) if lead_times else (float('nan'), float('nan'))

print(f"\nt+15 (FIXED): episodes={n_episodes}  event_recall={event_recall:.1f}% "
      f"(CI {ci_low*100:.1f}-{ci_high*100:.1f}%)  median_lead={median_lead:.1f}min (IQR {q1:.1f}-{q3:.1f})")
print(f"Previous (buggy) result was: 23.8% (CI 10.6-45.1%)")

# ── Also re-check sample-level recall/precision/FA-day with fixed tau ──
all_true = np.concatenate([d["true"] for d in t15_fixed_data])
all_single = np.concatenate([d["single_alarm"] for d in t15_fixed_data])
y_true = (all_true < 3.9).astype(int)
n_days = len(all_true) / 288
rec = recall_score(y_true, all_single, zero_division=0)*100
prec = precision_score(y_true, all_single, zero_division=0)*100
fa = ((all_single==1)&(y_true==0)).sum()/n_days
print(f"\nt+15 (FIXED) sample-level: Recall={rec:.1f}%  Precision={prec:.1f}%  FA/day={fa:.2f}")
print(f"Previous (buggy) sample-level was: Recall=80.7% Precision=89.5% FA/day=1.99")

In [ ]:
subject_dir = os.path.dirname(all_subjects[0]["glucose_csv"])
insulin_path = os.path.join(subject_dir, "insulin.csv")
insulin_df = pd.read_csv(insulin_path)
print("Insulin columns:", insulin_df.columns.tolist())
print(insulin_df.head())

In [ ]:
def add_meal_insulin_features_fixed(merged_df, subject_dir):
    """
    Handles the ACTUAL D1NAMO format: food.csv/insulin.csv have a single
    combined 'datetime' column in 'YYYY:MM:DD HH:MM:SS' format (colons as
    date separators), not separate 'date'/'time' columns.
    Falls back to the old date+time format if that's what's present (robustness).
    """
    for fname, col, out in [("food.csv", "carbs", "recent_carbs"),
                            ("insulin.csv", "units", "recent_units")]:
        fpath = os.path.join(subject_dir, fname)
        merged_df[out] = 0.0
        if not os.path.exists(fpath):
            continue
        df = pd.read_csv(fpath)

        # Case 1: actual D1NAMO format — combined 'datetime' column, colons for date part
        if "datetime" in df.columns:
            # format looks like "2014:10:01 19:27:49" -> replace first two colons with dashes
            def parse_dnamo_dt(s):
                try:
                    date_part, time_part = s.split(" ", 1)
                    date_fixed = date_part.replace(":", "-")
                    return pd.to_datetime(f"{date_fixed} {time_part}", errors="coerce")
                except Exception:
                    return pd.NaT
            df["datetime"] = df["datetime"].apply(parse_dnamo_dt)

        # Case 2: fallback — separate date/time columns (older assumption, kept for robustness)
        elif {"date", "time"}.issubset(df.columns):
            df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
        else:
            continue  # neither format found, skip

        # Figure out which column actually holds the carbs/insulin quantity —
        # 'calories' is present for food.csv (from the earlier inspection), not 'carbs'
        if col not in df.columns:
            # try common alternates seen in D1NAMO
            alt_names = {"carbs": ["calories", "carbohydrates", "carb"],
                        "units": ["units", "insulin", "dose"]}
            found = None
            for alt in alt_names.get(col, []):
                if alt in df.columns:
                    found = alt
                    break
            if found is None:
                continue
            col_to_use = found
        else:
            col_to_use = col

        df = df.dropna(subset=["datetime"]).sort_values("datetime")
        merged_df[out] = _rolling_2h_sum(df["datetime"], df[col_to_use], merged_df["datetime"])
    return merged_df

In [ ]:
# Test on one subject before rebuilding everything
test_dir = os.path.dirname(all_subjects[0]["glucose_csv"])
test_merged = subject_merged[subj_ids[0]].copy()
test_fixed = add_meal_insulin_features_fixed(test_merged, test_dir)
print("Non-zero carbs after fix:", (test_fixed["recent_carbs"] != 0).sum(), "/", len(test_fixed))
print("Non-zero units after fix:", (test_fixed["recent_units"] != 0).sum(), "/", len(test_fixed))
print(test_fixed[test_fixed["recent_carbs"] != 0][["datetime", "recent_carbs"]].head())

In [ ]:
def add_meal_insulin_features_fixed(merged_df, subject_dir):
    """
    food.csv: combined 'datetime' column, colon-separated date. Uses 'calories'
    as carb proxy (D1NAMO has no direct carb-gram column), SCALED DOWN to avoid
    dominating the regression due to magnitude mismatch with glucose (mmol/L).

    insulin.csv: separate 'date'+'time' columns. Uses fast_insulin + slow_insulin
    (summed, treating NaN as 0) as the insulin dose.
    """
    # ── FOOD / CARBS ──
    food_path = os.path.join(subject_dir, "food.csv")
    merged_df["recent_carbs"] = 0.0
    if os.path.exists(food_path):
        df = pd.read_csv(food_path)
        if "datetime" in df.columns:
            def parse_dnamo_dt(s):
                try:
                    date_part, time_part = s.split(" ", 1)
                    return pd.to_datetime(f"{date_part.replace(':', '-')} {time_part}", errors="coerce")
                except Exception:
                    return pd.NaT
            df["datetime"] = df["datetime"].apply(parse_dnamo_dt)
            df = df.dropna(subset=["datetime"]).sort_values("datetime")
            if "calories" in df.columns:
                # scale calories down (typical meal ~300-1500 kcal) to a rough 0-15 "carb-like" range
                # so it doesn't dominate regression purely due to magnitude
                carb_proxy = df["calories"].astype(float) / 100.0
                merged_df["recent_carbs"] = _rolling_2h_sum(df["datetime"], carb_proxy, merged_df["datetime"])

    # ── INSULIN ──
    insulin_path = os.path.join(subject_dir, "insulin.csv")
    merged_df["recent_units"] = 0.0
    if os.path.exists(insulin_path):
        df = pd.read_csv(insulin_path)
        if {"date", "time"}.issubset(df.columns):
            df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
            df = df.dropna(subset=["datetime"]).sort_values("datetime")
            fast = df["fast_insulin"].fillna(0).astype(float) if "fast_insulin" in df.columns else 0.0
            slow = df["slow_insulin"].fillna(0).astype(float) if "slow_insulin" in df.columns else 0.0
            total_units = fast + slow
            merged_df["recent_units"] = _rolling_2h_sum(df["datetime"], total_units, merged_df["datetime"])

    return merged_df

In [ ]:
test_dir = os.path.dirname(all_subjects[0]["glucose_csv"])
test_merged = subject_merged[subj_ids[0]].copy()
test_fixed = add_meal_insulin_features_fixed(test_merged, test_dir)
print("Non-zero carbs:", (test_fixed["recent_carbs"] != 0).sum(), "/", len(test_fixed),
     " max:", test_fixed["recent_carbs"].max())
print("Non-zero units:", (test_fixed["recent_units"] != 0).sum(), "/", len(test_fixed),
     " max:", test_fixed["recent_units"].max())

In [ ]:
# ── Rebuild subject_merged with fixed meal/insulin features ──
subject_merged_fixed = {}
for s in all_subjects:
    gdf = load_glucose(s["glucose_csv"]); sdf = load_sensor(s["sensor_path"])
    if sdf is None or len(gdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    merged = align_glucose_sensor(gdf, sdf)
    merged = add_meal_insulin_features_fixed(merged, os.path.dirname(s["glucose_csv"]))
    merged = add_cgm_derivative_features(merged)
    subject_merged_fixed[s["id"]] = merged

print("Rebuilt subject_merged_fixed for", len(subject_merged_fixed), "subjects")

# ── Check non-zero counts across ALL subjects (confirm fix is universal) ──
for s in subj_ids:
    m = subject_merged_fixed[s]
    c, u = m["recent_carbs"].values, m["recent_units"].values
    print(f"{s:16s}: nonzero_carbs={(c!=0).sum():4d}  nonzero_units={(u!=0).sum():4d}  "
          f"max_carb={c.max():6.2f}  max_unit={u.max():5.1f}")


# ── Refit the ORIGINAL physics formula (old linear-extrapolation, with k1/k2) using fixed data ──
def fit_old_physics_fixed(all_merged_train, h):
    X_list, y_list = [], []
    for merged in all_merged_train:
        g = merged["glucose"].values.astype(np.float64)
        roc = merged["cgm_roc"].values.astype(np.float64)
        carbs = merged["recent_carbs"].values.astype(np.float64)
        units = merged["recent_units"].values.astype(np.float64)
        for i in range(len(g) - h):
            target_delta = g[i + h] - (g[i] + roc[i] * h)
            X_list.append([carbs[i], units[i]])
            y_list.append(target_delta)
    X = np.array(X_list); y = np.array(y_list)
    reg = LinearRegression().fit(X, y)
    return {"k1": reg.coef_[0], "k2": reg.coef_[1], "b": reg.intercept_}


print("\n" + "="*60)
print("k1/k2 REFIT WITH FIXED MEAL/INSULIN DATA")
print("="*60)
all_merged_fixed = [subject_merged_fixed[s] for s in subj_ids]
for h, label in zip(HORIZONS, ["t+15", "t+30", "t+60"]):
    coefs = fit_old_physics_fixed(all_merged_fixed, h)
    print(f"{label}: k1(carbs)={coefs['k1']:.5f}  k2(units)={coefs['k2']:.5f}  b={coefs['b']:.5f}")

In [ ]:
def add_meal_insulin_features_fixed(merged_df, subject_dir):
    # ── FOOD / CARBS ──
    food_path = os.path.join(subject_dir, "food.csv")
    merged_df["recent_carbs"] = 0.0
    if os.path.exists(food_path):
        df = pd.read_csv(food_path)
        if "datetime" in df.columns:
            def parse_dnamo_dt(s):
                try:
                    date_part, time_part = s.split(" ", 1)
                    return pd.to_datetime(f"{date_part.replace(':', '-')} {time_part}", errors="coerce")
                except Exception:
                    return pd.NaT
            df["datetime"] = df["datetime"].apply(parse_dnamo_dt)
            df = df.dropna(subset=["datetime"]).sort_values("datetime")
            if "calories" in df.columns:
                carb_proxy = df["calories"].fillna(0).astype(float) / 100.0   # FIX: fillna before use
                merged_df["recent_carbs"] = _rolling_2h_sum(df["datetime"], carb_proxy, merged_df["datetime"])

    # ── INSULIN ──
    insulin_path = os.path.join(subject_dir, "insulin.csv")
    merged_df["recent_units"] = 0.0
    if os.path.exists(insulin_path):
        df = pd.read_csv(insulin_path)
        if {"date", "time"}.issubset(df.columns):
            df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
            df = df.dropna(subset=["datetime"]).sort_values("datetime")
            fast = df["fast_insulin"].fillna(0).astype(float) if "fast_insulin" in df.columns else 0.0
            slow = df["slow_insulin"].fillna(0).astype(float) if "slow_insulin" in df.columns else 0.0
            total_units = fast + slow
            merged_df["recent_units"] = _rolling_2h_sum(df["datetime"], total_units, merged_df["datetime"])

    return merged_df

In [ ]:
subject_merged_fixed = {}
for s in all_subjects:
    gdf = load_glucose(s["glucose_csv"]); sdf = load_sensor(s["sensor_path"])
    if sdf is None or len(gdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    merged = align_glucose_sensor(gdf, sdf)
    merged = add_meal_insulin_features_fixed(merged, os.path.dirname(s["glucose_csv"]))
    merged = add_cgm_derivative_features(merged)
    subject_merged_fixed[s["id"]] = merged

print("Rebuilt subject_merged_fixed for", len(subject_merged_fixed), "subjects")

# Re-verify no NaN this time
for s in subj_ids:
    m = subject_merged_fixed[s]
    c, u = m["recent_carbs"].values, m["recent_units"].values
    print(f"{s:16s}: nonzero_carbs={(c!=0).sum():4d}  nonzero_units={(u!=0).sum():4d}  "
          f"max_carb={np.nanmax(c) if not np.isnan(c).all() else 0:6.2f}  "
          f"max_unit={u.max():5.1f}  any_nan_carb={np.isnan(c).any()}")

# ── Refit k1/k2 ──
all_merged_fixed = [subject_merged_fixed[s] for s in subj_ids]
print("\n" + "="*60)
print("k1/k2 REFIT WITH FIXED MEAL/INSULIN DATA")
print("="*60)
for h, label in zip(HORIZONS, ["t+15", "t+30", "t+60"]):
    coefs = fit_old_physics_fixed(all_merged_fixed, h)
    print(f"{label}: k1(carbs)={coefs['k1']:.5f}  k2(units)={coefs['k2']:.5f}  b={coefs['b']:.5f}")

In [ ]:
# Check correlation between carbs and units across all subjects
all_carbs, all_units = [], []
for s in subj_ids:
    m = subject_merged_fixed[s]
    all_carbs.extend(m["recent_carbs"].values)
    all_units.extend(m["recent_units"].values)

all_carbs, all_units = np.array(all_carbs), np.array(all_units)
corr = np.corrcoef(all_carbs, all_units)[0, 1]
print(f"Correlation between recent_carbs and recent_units: {corr:.3f}")

In [ ]:
# Quick test: does the old physics formula (with fixed carbs/units) improve over the version without them?
def evaluate_old_physics_fixed():
    all_P, all_T = {h: [] for h in HORIZONS}, {h: [] for h in HORIZONS}
    for held_out in subj_ids:
        train_subjs = [s for s in subj_ids if s != held_out]
        train_merged = [subject_merged_fixed[s] for s in train_subjs]
        merged = subject_merged_fixed[held_out]
        g = merged["glucose"].values.astype(np.float64)
        roc = merged["cgm_roc"].values.astype(np.float64)
        carbs = merged["recent_carbs"].values.astype(np.float64)
        units = merged["recent_units"].values.astype(np.float64)
        max_h = max(HORIZONS)
        for h in HORIZONS:
            coefs = fit_old_physics_fixed(train_merged, h)
            for i in range(len(g) - max_h):
                p = g[i] + roc[i]*h + coefs["k1"]*carbs[i] + coefs["k2"]*units[i] + coefs["b"]
                all_P[h].append(p)
                all_T[h].append(g[i+h])
    return all_P, all_T

P_old_fixed, T_old_fixed = evaluate_old_physics_fixed()
print("Old physics formula WITH fixed carbs/units:")
for h, label in zip(HORIZONS, ["t+15", "t+30", "t+60"]):
    p, t = np.array(P_old_fixed[h]), np.array(T_old_fixed[h])
    print(f"{label}: MARD = {mard(t, p):.2f}%   (compare to old-broken: 4.10%/11.15%/28.57%)")

In [ ]:
# ── Corrected S4D verdict: physics_pred stored alongside hybrid predictions ──

def run_losocv_tuned_with_physics_stored(n_epochs=60, patience=12, ft_epochs=4, ft_lr_scale=0.03, ft_frac=0.3):
    all_preds, all_trues, all_physics = [], [], []

    for held_out in subj_ids:
        train_subjs = [s for s in subj_ids if s != held_out]
        v3_coefs = {h: fit_empirical_formula_v3([subject_merged[s] for s in train_subjs], h) for h in HORIZONS}

        train_recs, val_recs = [], []
        for s in train_subjs:
            recs = build_windows_hybrid_v3(subject_merged[s], v3_coefs)
            split = int(len(recs) * 0.9)
            train_recs.extend(recs[:split]); val_recs.extend(recs[split:])
        test_recs_all = build_windows_hybrid_v3(subject_merged[held_out], v3_coefs)
        ft_split = max(WINDOW_SIZE, int(len(test_recs_all) * ft_frac))
        ft_recs, eval_recs = test_recs_all[:ft_split], test_recs_all[ft_split:]
        if len(eval_recs) < 5:
            continue

        n_sensors = len(train_recs[0]["sensor_window"][0])
        tr_loader = DataLoader(D1NAMODataset(train_recs, augment=True, use_physics=True),
                               batch_size=32, shuffle=True, drop_last=True)
        va_loader = DataLoader(D1NAMODataset(val_recs, augment=False, use_physics=True),
                               batch_size=32, shuffle=False)
        ft_loader = DataLoader(D1NAMODataset(ft_recs, augment=False, use_physics=True),
                               batch_size=16, shuffle=True)
        ev_loader = DataLoader(D1NAMODataset(eval_recs, augment=False, use_physics=True),
                               batch_size=32, shuffle=False)

        torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
        m = D1NAMOPhysicsInformed_TunedConstrained(n_sensors=n_sensors).to(device)
        for mod in m.modules():
            if isinstance(mod, S4DKernel):
                for p in [mod.log_dt, mod.log_A_real, mod.A_imag, mod.C]:
                    p._is_ssm = True

        ssm   = [p for p in m.parameters() if getattr(p, "_is_ssm", False)]
        other = [p for p in m.parameters() if not getattr(p, "_is_ssm", False)]
        opt = torch.optim.AdamW([{"params": other, "lr": 5e-4, "weight_decay": 1e-2},
                                 {"params": ssm, "lr": 1e-3, "weight_decay": 0.0}])
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-6)

        ckpt = os.path.join(OUTPUT_DIR, f"tuned_verify_{held_out}.pt")
        best, no_imp = float("inf"), 0
        for ep in range(n_epochs):
            run_epoch_weighted(m, tr_loader, opt)
            va = run_epoch_weighted(m, va_loader)
            sch.step()
            if va < best:
                best, no_imp = va, 0
                torch.save(m.state_dict(), ckpt)
            else:
                no_imp += 1
            if no_imp >= patience:
                break
        m.load_state_dict(torch.load(ckpt, map_location=device))

        ft_opt = torch.optim.AdamW([{"params": other, "lr": 5e-4 * ft_lr_scale, "weight_decay": 1e-2},
                                    {"params": ssm, "lr": 1e-3 * ft_lr_scale, "weight_decay": 0.0}])
        for ep in range(ft_epochs):
            run_epoch_weighted(m, ft_loader, ft_opt)

        m.eval(); P, T, PHYS = [], [], []
        with torch.no_grad():
            for cgm, sensor, y, phys in ev_loader:
                pred = m(cgm.to(device), sensor.to(device), phys.to(device))
                P.append(pred.cpu().numpy()); T.append(y.numpy())
                PHYS.append(phys.numpy())   # NEW: store the exact physics_pred used for THIS sample
        P, T, PHYS = np.concatenate(P), np.concatenate(T), np.concatenate(PHYS)
        gm, gs = eval_recs[0]["glucose_mean"], eval_recs[0]["glucose_std"]
        P_real, T_real, PHYS_real = P*gs+gm, T*gs+gm, PHYS*gs+gm   # denormalize physics_pred too
        all_preds.append(P_real); all_trues.append(T_real); all_physics.append(PHYS_real)
        maes = [mean_absolute_error(T_real[:, i], P_real[:, i]) for i in range(3)]
        print(f"  {held_out:16s} | MAE 15/30/60: {maes[0]:.3f}/{maes[1]:.3f}/{maes[2]:.3f}")

    return np.concatenate(all_preds), np.concatenate(all_trues), np.concatenate(all_physics)


print("Running LOSOCV with physics_pred stored (for correction-magnitude verification)...")
P_verify, T_verify, PHYS_verify = run_losocv_tuned_with_physics_stored()

# ── Now correction magnitude is exact — same samples, guaranteed alignment ──
print("\n" + "="*60)
print("S4D CORRECTION MAGNITUDE (CORRECTED — exact sample alignment)")
print("="*60)
for i, h in enumerate(["t+15", "t+30", "t+60"]):
    corr = P_verify[:, i] - PHYS_verify[:, i]
    print(f"{h}: mean|correction| = {np.abs(corr).mean():.4f} mmol/L, "
          f"max|correction| = {np.abs(corr).max():.4f} mmol/L, "
          f"std = {corr.std():.4f} mmol/L")

# Sanity check: correction should be bounded (model uses tanh * correction_scales, max 20%)
print("\nSanity check — max correction should be small (bounded architecture):")
for i, h in enumerate(["t+15", "t+30", "t+60"]):
    corr = P_verify[:, i] - PHYS_verify[:, i]
    phys_range = PHYS_verify[:, i].max() - PHYS_verify[:, i].min()
    print(f"{h}: max|correction|={np.abs(corr).max():.3f}  (physics range for context: {phys_range:.1f})")